<a href="https://colab.research.google.com/github/Fantiflex/Cognitive_biaises/blob/main/Image_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install opencv-python numpy face-alignment torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.4 MB/s eta 0:00:00


# **Imports**

In [2]:
import cv2
import numpy as np
import face_alignment
import os


#Chargement du modèle de Face Parsing

On utilise un réseau de neurones appelé HRNet (Comprehensive facial analysis by extracting face features).

In [3]:
# Le paramètre face_detector='blazeface' ou 'sfd' localise le visage,
# puis HRNet applique la prédiction haute résolution des points clés.
print("Chargement du modèle HRNet...")
fa = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.TWO_D,
    device='cuda',
    face_detector='sfd'
)

Chargement du modèle HRNet...
Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth


100%|██████████| 85.7M/85.7M [00:11<00:00, 7.96MB/s]
Downloading: "https://www.adrianbulat.com/downloads/python-fan/2DFAN4-11f355bf06.pth.tar" to /root/.cache/torch/hub/checkpoints/2DFAN4-11f355bf06.pth.tar
100%|██████████| 91.2M/91.2M [00:07<00:00, 13.6MB/s]
/usr/local/lib/python3.12/dist-packages/face_alignment/api.py:130: UserWarning: Compiling face alignment model (one-time cost). Subsequent runs will be faster.
  warnings.warn(
W0702 14:14:16.172000 691 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


# **Helper functions**

In [4]:
def extract_luminance_hrnet(img):
    # Charger l'image
    if img is None:
        return None

    h, w, _ = img.shape

    # 2. Détection des points clés via HRNet (attend du RGB)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    preds = fa.get_landmarks(img_rgb)

    if preds is None or len(preds) == 0:
        print(f"Aucun visage détecté par HRNet dans : {image_path}")
        return None

    # Récupérer les coordonnées du premier visage détecté
    landmarks = preds[0] # Tableau de 68 points (x, y)

    # 3. Construction du contour de la peau (Exclusion des cheveux et du fond)
    # Les points 0 à 16 décrivent la mâchoire (du haut de l'oreille gauche à l'oreille droite)
    jawline = landmarks[0:17]

    # Les points 17 à 26 décrivent les sourcils.
    # On les utilise pour fermer le haut du visage (le front) sans mordre sur les cheveux.
    eyebrows = landmarks[17:27]

    # On combine la mâchoire et les sourcils pour former le polygone du visage "nu"
    # Pour un front légèrement plus haut, on peut appliquer un décalage vertical (offset) sur les sourcils
    face_contour = np.concatenate((jawline, eyebrows[::-1]), axis=0).astype(np.int32)

    # 4. Création du masque binaire
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [face_contour], 255)

    # 5. Conversion dans l'espace LAB pour la Luminance
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
    L_channel, _, _ = cv2.split(img_lab)

    # Isoler les pixels du visage
    face_pixels = L_channel[mask == 255]

    if len(face_pixels) == 0:
        return None

    # Calcul de la luminance moyenne (Échelle 0 - 255 dans OpenCV)
    mean_luminance = np.mean(face_pixels)
    return mean_luminance


Espace LAB : L'espace de couleur CIE L*a*b* est un modèle colorimétrique standardisé par la Commission Internationale de l'Éclairage (CIE) en 1976.

Contrairement aux espaces RVB (RGB) ou CMJN, qui dépendent du matériel un écran ou une imprimante spécifique, l'espace LAB a été conçu pour être indépendant des appareils et pour correspondre à la perception visuelle humaine.